# Node Class

In [13]:
class Node:
    def __init__(self, value, next_node=None):
        self.value = value
        self.next = next_node

    def __repr__(self):
        return f"Node({self.value!r})"

    def __str__(self):
            return f"Node({self.value!r})"

In [16]:
a = Node(12)
print(a)

b = Node(2, a)
print(b, "->", b.next)

c = Node(4, Node(5, b))
print(c, "->", c.next, "->", c.next.next, "->", c.next.next.next, "->", c.next.next.next.next)
# Node(4) -> Node(5) -> Node(2) -> Node(12) -> None

sep = " -> "
strings = [str(c), str(c.next), str(c.next.next), str(c.next.next.next), str(c.next.next.next.next)]
print(strings)
print(sep.join(strings))

Node(12)
Node(2) -> Node(12)
Node(4) -> Node(5) -> Node(2) -> Node(12) -> None
['Node(4)', 'Node(5)', 'Node(2)', 'Node(12)', 'None']
Node(4) -> Node(5) -> Node(2) -> Node(12) -> None


## Additional Preparations

In [8]:
class EmptyListError(IndexError):
    """Access to an empty data structure.
    
    Inherits from IndexError: whoever writes 'except IndexError' will
    automatically catch this excpetion as well."""
    pass

class OperationNotSupportedError(TypeError):
    """A subclass forbids an operation it has inherited."""

# Linked List

In [19]:
class LinkedList:
    separator = " -> "

    def __init__(self, iterable=None):
        self._head = None
        self._tail = None
        self._size = 0
        if iterable is not None:
            for value in iterable:
                self.add(value)

    def add(self, value):
        self.append(value)

    def take(self):
        return self.remove_first()

    def peek(self):
        if self.is_empty():
            raise EmptyListError(f"peek() on empty {type(self).__name__}")
        return self._head.value

    def append(self, value):
        node = Node(value)
        if self._tail is None:
            self._head = self._tail = node
        else:
            self._tail.next = node
            self._tail = node
        self._size += 1  # self._size = self._size + 1

    def prepend(self, value):
        node = Node(value, self._head)
        self._head = node
        if self._tail is None:
            self._tail = node
        self._size += 1

    def insert(self, index, value):
        if not 0 <= index <= self._size:
            raise IndexError(f"Index {index} outside of 0..{self._size}")

        if index == 0:
            self.prepend(value)
        elif index == self._size:
            self.append(value)
        else:
            previous = self._node_at(index - 1)
            following = previous.next
            previous.next = Node(value, following)
            self._size += 1


    def remove_first(self):
        if self._head is None:
            raise EmptyListError(f"remove_first() on empty {type(self).__name__}")
        node = self._head
        self._head = node.next
        if self._head is None:  # in case new list is empty
            self._tail = None
        self._size -= 1
        return node.value

    def remove_last(self):
        if self._head is None:
            raise EmptyListError(f"remove_last() on empty {type(self).__name__}")
        if self._head is self._tail:
            return self.remove_first()
        previous = self._node_at(self._size - 2)
        last = previous.next
        previous.next = None
        self._tail = previous
        self._size -= 1
        return last.value

    def remove(self, value):
        previous = None
        current = self._head
        while current is not None:
            if current.value == value:
                if previous is None:
                    self._head = current.next
                else:
                    previous.next = current.next
                if current is self._tail:
                    self._tail = previous
                self._size -= 1
                return
            previous = current
            current = current.next
        raise ValueError(f"{value!r} is not in the list")

    def clear(self):
        self._head = self._tail = None
        self._size = 0

    def is_empty(self):
        return self._size == 0

    def get(self, index):
        if index < 0:
            index += self._size
        if not 0 <= index < self._size:
            raise IndexError(f"Index {index} outside of 0..{self._size - 1 }")
        return self._node_at(index).value

    def index_of(self, value):
        for i, item in enumerate(self):
            if item == value:
                return i
        raise ValueError(f"{value!r} is not in the list")

    def _node_at(self, index):
        current = self._head
        for _ in range(index):        # while index > 0:
            current = current.next    #     current = current.next
                                      #     index -= 1
        return current

    def __len__(self):          # len(list)
        return self._size

    def __iter__(self):
        current = self._head
        while current is not None:
            yield current.value
            current = current.next

    def __contains__(self, value):
        return any(item == value for item in self)
        """long way
            result = false
            for item in self:
                result = result or (item == value)
            return result
        """
    def __getitem__(self, index):       # list[i]
        return self.get(index)

    def __eq__(self, other):     # list1 == list2
        if type(self) is not type(other):
            return NotImplemented

        return list(self) == list(other)
        """
        Otherwise:
        1. check is size is same
        2. check if head and tail have the same values
        3. iterate over all nodes together and check if nodes have same values at same index
        """

    def __str__(self):
        items = self.separator.join(str(v) for v in self)
        return f"{type(self).__name__}[{items}]"

    def __repr__(self):
        return f"{type(self).__name__}({list(self)!r})"

    def _forbid(self, operation):
        raise OperationNotSupportedError(f"{type(self).__name__} does not support '{operation}'")

In [ ]:
lst = LinkedList([1, 2, 3])
print(lst)
lst.insert(2, 1.5)
print(lst)  # uses __str__
lst         # uses __repr__

LinkedList[1 -> 2 -> 3]
LinkedList[1 -> 2 -> 1.5 -> 3]


LinkedList([1, 2, 1.5, 3])

# Stack - LIFO / FILO
With preparation first for new Error handling

In [ ]:
class FullStackError(OverflowError):
    """push() onto a full stack."""

In [ ]:
class Stack(LinkedList):
    separator = " | "   # override of a class attribute

    def __init__(self, iterable=None, capacity=None):
        # Mind the order! capacity MUST be set before super().__init__,
        # because LinkedList.__init__ calls self.add() -> self.push(),
        # and push() already reads self.capacity.
        self.capacity = capacity
        super().__init__(iterable)

    # --- Stack-specific methods ---------------------------------------
    def push(self, value):
        if self.capacity is not None and len(self) >= self.capacity:
            raise FullStackError(f"Stack is full (capacity={self.capacity})")
        self.prepend(value)

    def pop(self):
        if self.is_empty():
            raise EmptyListError("pop() on empty Stack")
        return self.remove_first()

    # peek() is NOT overridden: LinkedList.peek returns _head,
    # which already is the top of our stack.

    # --- Overrides of the generic interface ---------------------------
    def add(self, value):
        self.push(value)

    def take(self):
        return self.pop()

    # --- Forbidden operations (would violate LIFO) --------------------
    def append(self, value):
        self._forbid("append")

    def insert(self, index, value):
        self._forbid("insert")

    def remove(self, value):
        self._forbid("remove")

    def remove_last(self):
        self._forbid("remove_last")

    # --- Representation ------------------------------------------------
    def __str__(self):
        # Override that REUSES the base implementation via super()
        return super().__str__() + "  (top = left)"

    def __repr__(self):
        # This override is required to keep the output correct:
        # iteration runs from top to bottom, but Stack([1, 2, 3]) pushes
        # 1, then 2, then 3. We therefore output the elements reversed,
        # so that eval(repr(s)) == s holds.
        return f"Stack({list(reversed(list(self)))!r})"

In [ ]:
s = Stack()
for x in "abc":
    s.push(x)
print(s)
print("pop():", s.pop(), "| peek():", s.peek(), "->", s)

try:
    s.append("x")       # inherited, but blocked
except OperationNotSupportedError as e:
    print(f"{type(e).__name__}: {e}")

# Queue - FIFO / LILO

In [ ]:
class Queue(LinkedList):
    separator = " <- "  # override of a class attribute

    # No own __init__ needed -> LinkedList.__init__ is inherited.

    # --- Queue-specific methods ---------------------------------------
    def enqueue(self, value):
        self.append(value)

    def dequeue(self):
        if self.is_empty():
            raise EmptyListError("dequeue() on empty Queue")
        return self.remove_first()

    def front(self):
        # Call of the base implementation under a new name
        return super().peek()

    # --- Override of the generic interface ----------------------------
    # add() is inherited (append == enqueue).
    def take(self):
        return self.dequeue()

    # --- Forbidden operations (would violate FIFO) --------------------
    def prepend(self, value):
        self._forbid("prepend")

    def insert(self, index, value):
        self._forbid("insert")

    def remove(self, value):
        self._forbid("remove")

    def remove_last(self):
        self._forbid("remove_last")

    # --- Representation ------------------------------------------------
    def __str__(self):
        return f"front {super().__str__()} back"

In [ ]:
q = Queue()
for x in "abc":
    q.enqueue(x)
print(q)
print("dequeue():", q.dequeue(), "| front():", q.front(), "->", q)

# Comparing and checking Inheritance and Polymorphism

In [ ]:
for container in (LinkedList([1, 2, 3]), Stack([1, 2, 3]), Queue([1, 2, 3])):
    container.add(4)
    taken = [container.take() for _ in range(len(container))]
    print(f"{type(container).__name__:<10} takes: {taken}")

In [ ]:
s = Stack([1, 2, 3])
print("isinstance(s, Stack)?      ", isinstance(s, Stack))
print("isinstance(s, LinkedList)? ", isinstance(s, LinkedList))
print("issubclass(Queue, Stack)?  ", issubclass(Queue, Stack))
print("MRO of Stack:", [cls.__name__ for cls in Stack.__mro__])
print("Stack.separator:", repr(Stack.separator),
      "| LinkedList.separator:", repr(LinkedList.separator))
print("Stack([1]) == Queue([1])?  ", Stack([1]) == Queue([1]))
print("eval(repr(Stack([1,2,3]))) == Stack([1,2,3])?",
      eval(repr(Stack([1, 2, 3]))) == Stack([1, 2, 3]))

In [ ]:
try:
    Stack().pop()
except IndexError as e:                 # also catches EmptyListError!
    print(f"{type(e).__name__}: {e}")

try:
    full = Stack([1, 2], capacity=2)
    full.push(3)
except FullStackError as e:
    print(f"{type(e).__name__}: {e}")

try:
    Queue().prepend("x")                # inherited, but blocked in Queue
except OperationNotSupportedError as e:
    print(f"{type(e).__name__}: {e}")

try:
    Queue().dequeue()
except EmptyListError as e:
    print(f"{type(e).__name__}: {e}")